# Exploração: escolha do modelo de classificação

**Objetivo:** escolher o modelo que classifica cada transação (CSV do parser) em uma das 8 categorias,
justificando a escolha com experimentos.

**Como ler os resultados**
- **Acurácia**: % de transações classificadas certo.
- **F1 macro**: média do F1 de cada classe — dá o mesmo peso a classes pequenas (ex.: `transporte`).
- **Validação aleatória** (`StratifiedKFold`, 5 folds): transações sorteadas entre treino e teste.
  O mesmo usuário e as mesmas lojas aparecem dos dois lados → resultado **otimista**.
- **Validação por usuário** (`GroupKFold`, 5 folds): todas as transações de um usuário ficam juntas no
  treino **ou** no teste → mede o caso real (usuário novo na aplicação). **É a métrica usada para decidir.**

Em todos os testes o pré-processamento fica dentro de um `Pipeline`: é ajustado só com o treino de cada fold
(sem vazar informação do teste).

> Antes de fazer commit, limpe as saídas (**Clear All Outputs**): elas podem conter dados derivados dos extratos.

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, StratifiedKFold, cross_validate
from sklearn.naive_bayes import ComplementNB, GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.precision", 3)

## 1. Dados

In [ ]:
# funciona com o notebook aberto em ml/ ou na raiz do projeto
CSV = next(p for p in (Path("dados/transacoes_rotuladas.csv"), Path("ml/dados/transacoes_rotuladas.csv")) if p.exists())

# keep_default_na=False: contraparte vazia continua "" (sem contraparte), não vira NaN
df = pd.read_csv(CSV, keep_default_na=False, parse_dates=["data"])
y = df["categoria"]
grupos = df["id_usuario"]  # não é atributo do modelo: só define os grupos da validação por usuário

print(f"{len(df)} transações | {grupos.nunique()} usuários | {y.nunique()} classes")
y.value_counts().to_frame("transações").assign(**{"%": lambda t: (100 * t["transações"] / len(df)).round(1)})

## 2. Atributos e normalização

| Coluna do parser | Vira | Normalização |
|---|---|---|
| `historico`, `meio`, `tipo`, `banco` | categóricas | **one-hot** (uma coluna 0/1 por valor; valores com < 5 ocorrências viram "infrequent") |
| `contraparte` (nome da loja) | categórica **ou** texto | **one-hot** ou **TF-IDF de caracteres** (comparados na etapa 4) |
| `valor` | `valor_log = log(1 + |valor|)` | **min-max** para [0, 1]; o sinal já está em `tipo`, o log reduz a cauda longa |
| `data` | `dia_semana`, `dia_mes`, `mes` | **min-max** (testados com e sem na etapa 4) |
| `id_usuario` | — | **não entra** (usuários futuros não estão no treino): só agrupa a validação |

In [ ]:
CATEGORICAS = ["historico", "meio", "tipo", "banco"]
COLUNAS_DATA = ["dia_semana", "dia_mes", "mes"]

feat = df[CATEGORICAS + ["contraparte"]].copy()
feat["valor_log"] = np.log1p(df["valor"].abs())
feat["dia_semana"] = df["data"].dt.day_of_week
feat["dia_mes"] = df["data"].dt.day
feat["mes"] = df["data"].dt.month


def preprocessamento(contraparte="onehot", data=True, denso=False):
    """ColumnTransformer: one-hot nas categóricas, min-max nas numéricas e contraparte em one-hot ou TF-IDF."""
    numericas = ["valor_log"] + (COLUNAS_DATA if data else [])
    onehot = lambda: OneHotEncoder(min_frequency=5, handle_unknown="infrequent_if_exist", sparse_output=denso)
    partes = [("cat", onehot(), CATEGORICAS), ("num", MinMaxScaler(), numericas)]
    if contraparte == "onehot":
        partes.append(("contraparte", onehot(), ["contraparte"]))
    else:  # TF-IDF de pedaços de 3 a 5 letras: reconhece lojas novas pelo que já viu
        partes.append(("contraparte", TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5), min_df=3,
                                                      sublinear_tf=True), "contraparte"))
    return ColumnTransformer(partes, sparse_threshold=0 if denso else 0.3)


print("colunas com one-hot:", preprocessamento(denso=True).fit_transform(feat).shape[1])
print("colunas com TF-IDF na contraparte:", preprocessamento("tfidf").fit_transform(feat).shape[1])

## 3. Validação

In [ ]:
CV_ALEATORIO = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
CV_USUARIO = GroupKFold(n_splits=5)


def avalia(modelo, X=feat):
    """Acurácia e F1 macro médios nas duas validações (o pré-processamento está dentro do modelo)."""
    r = {}
    for nome, cv, g in (("aleatório", CV_ALEATORIO, None), ("por usuário", CV_USUARIO, grupos)):
        s = cross_validate(modelo, X, y, cv=cv, groups=g, scoring=["accuracy", "f1_macro"], n_jobs=-1)
        r[f"acurácia ({nome})"] = s["test_accuracy"].mean()
        r[f"F1 macro ({nome})"] = s["test_f1_macro"].mean()
        if nome == "por usuário":
            r["± F1 (por usuário)"] = s["test_f1_macro"].std()
    return r


def tabela(resultados):
    return pd.DataFrame(resultados).T.sort_values("F1 macro (por usuário)", ascending=False)

## 4. Etapa 1 — comparação de modelos (one-hot + min-max, com data)

Mesma representação para todos; muda só o classificador. O one-hot é denso aqui porque o
`GaussianNB` e o `HistGradientBoosting` não aceitam matriz esparsa.

In [ ]:
modelos = {
    "SVM linear": LinearSVC(),
    "Regressão logística": LogisticRegression(max_iter=2000),
    "Naive Bayes (Complement)": ComplementNB(),
    "Naive Bayes (Gaussiano)": GaussianNB(),
    "KNN (k=5)": KNeighborsClassifier(n_neighbors=5),
    "Árvore de decisão": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Extra Trees": ExtraTreesClassifier(random_state=42),
    "HistGradientBoosting": HistGradientBoostingClassifier(random_state=42),  # o mais lento (~1-2 min)
}

etapa1 = {}
for nome, clf in modelos.items():
    inicio = time.perf_counter()
    etapa1[nome] = avalia(Pipeline([("prep", preprocessamento(denso=True)), ("clf", clf)]))
    etapa1[nome]["tempo (s)"] = time.perf_counter() - inicio
    print(f"{nome}: ok")
tabela(etapa1)

**Leitura (execução de 06/11/2026):**

| Modelo | F1 macro por usuário | Acurácia por usuário | Acurácia aleatória |
|---|---|---|---|
| **SVM linear** | **0,717** | **0,774** | 0,899 |
| Regressão logística | 0,686 | 0,741 | 0,898 |
| HistGradientBoosting | 0,676 | 0,746 | 0,869 |
| Extra Trees | 0,674 | 0,752 | 0,896 |
| Random Forest | 0,664 | 0,740 | 0,896 |
| Árvore de decisão | 0,634 | 0,730 | 0,876 |
| Naive Bayes (Complement) | 0,607 | 0,695 | 0,876 |
| KNN | 0,574 | 0,659 | 0,885 |
| Naive Bayes (Gaussiano) | 0,539 | 0,603 | 0,814 |

- **Os modelos lineares (SVM e regressão logística) generalizam melhor** para usuários novos: com centenas de colunas 0/1 e poucas linhas, um hiperplano separa bem as classes sem decorar detalhes de cada usuário.
- **Todos caem muito do aleatório para o por usuário** (~0,90 → ~0,75 de acurácia): com one-hot, uma loja nunca vista vira só "infrequent" e o modelo perde a informação mais importante.
- **O Gaussiano é o pior**: supõe que cada coluna segue uma curva normal, o que não vale para colunas 0/1.
- Seguem para a etapa 2 os três melhores de famílias diferentes: **SVM linear, regressão logística e Random Forest**.

## 5. Etapa 2 — representação: one-hot × TF-IDF na contraparte, com e sem data

Para os 3 melhores da etapa 1, muda só o pré-processamento.
- **TF-IDF de caracteres** na `contraparte`: o nome da loja vira pedaços de 3 a 5 letras
  (`PADARIA` → `PAD`, `ADA`, `DARIA`…). Uma loja nunca vista é reconhecida pelos pedaços em comum.
- **Sem data**: tira `dia_semana`, `dia_mes` e `mes`.

In [ ]:
etapa2 = {}
for nome in ["SVM linear", "Regressão logística", "Random Forest"]:
    for contraparte in ("onehot", "tfidf"):
        for data in (True, False):
            rotulo = f"{nome} | {contraparte} | {'com' if data else 'sem'} data"
            etapa2[rotulo] = avalia(Pipeline([("prep", preprocessamento(contraparte, data)),
                                              ("clf", modelos[nome])]))
            print(f"{rotulo}: ok")
tabela(etapa2)

**Leitura:**

| Configuração | F1 macro por usuário | Acurácia por usuário | Acurácia aleatória |
|---|---|---|---|
| **SVM linear + TF-IDF, sem data** | **0,903** ± 0,048 | **0,928** | **0,987** |
| SVM linear + TF-IDF, com data | 0,901 | 0,928 | 0,987 |
| Regressão logística + TF-IDF | 0,886–0,887 | 0,916–0,918 | 0,980 |
| Random Forest + TF-IDF | 0,856–0,857 | 0,911–0,913 | 0,984 |
| SVM linear + one-hot | 0,717–0,720 | 0,774–0,778 | 0,899 |

- **O TF-IDF na contraparte é a maior melhoria do projeto**: o SVM sobe de 0,72 para **0,90** de F1 macro por usuário. O modelo passa a reconhecer lojas novas pelos pedaços do nome (`SUPERMERC…`, `LANCH…`, `POSTO…`).
- **A data não ajuda** (diferença ≤ 0,002, dentro da variação): a categoria depende de *onde* e *como* foi a compra, não do dia. Sai do modelo — fica mais simples.
- **O SVM continua o melhor e o mais estável** (menor ± entre usuários).

## 6. Etapa 3 — ajuste do SVM (TF-IDF, sem data)

`C` controla a regularização (menor = modelo mais simples); `class_weight="balanced"` dá mais peso às
classes menores.

In [ ]:
etapa3 = {}
for C in (0.1, 0.3, 1.0, 3.0):
    for peso in (None, "balanced"):
        rotulo = f"C={C} | class_weight={peso}"
        etapa3[rotulo] = avalia(Pipeline([("prep", preprocessamento("tfidf", data=False)),
                                          ("clf", LinearSVC(C=C, class_weight=peso))]))
tabela(etapa3)

**Leitura:** todas as combinações ficam entre **0,895 e 0,906** de F1 macro por usuário — diferenças menores que a variação entre usuários (± ~0,05). O ajuste não muda o resultado de forma significativa; fica o **padrão (`C=1`, sem `class_weight`)**, o mais simples e com a melhor acurácia aleatória (0,987). `C=3` (menos regularização) é o único que piora de forma consistente.

## 7. Conclusão: modelo escolhido

**SVM linear + TF-IDF de caracteres na `contraparte`**, one-hot em `historico`/`meio`/`tipo`/`banco`, `log + min-max` no valor, **sem data**, `C=1`.

| Validação | Acurácia | F1 macro |
|---|---|---|
| Por usuário (caso real) | **0,928** | **0,903** |
| Aleatória | 0,987 | 0,988 |

**Por quê:**
1. **Melhor resultado em todas as etapas**, principalmente na validação por usuário, que é a que mede a aplicação real.
2. **Generaliza para lojas novas** graças ao TF-IDF de caracteres — o maior ganho do projeto (+0,18 de F1).
3. **Rápido** (segundos para treinar) e **interpretável**: os pesos mostram o que define cada classe (ver `champion_model.ipynb`).
4. **Estável**: menor variação entre usuários.

**Limitações conhecidas:**
- `compras` funciona como "classe de dúvida": quando não reconhece a loja, o modelo tende a ela.
- O histórico é diferente em cada banco; usuários com banco único nos dados (Inter, Santander, Sicoob) têm resultado pior.
- Só 8 usuários: os números variam bastante com quem está no teste.
- Parte dos rótulos de compras veio de palavras-chave no nome da loja, que o TF-IDF reaprende com facilidade.

O modelo escolhido é tratado isoladamente em **`champion_model.ipynb`**.